In [ ]:
""" | Feature            | `.load()`                       | `.lazy_load()`                   |
| ------------------ | ------------------------------- | -------------------------------- |
| Return type        | `list`                          | `generator`                      |
| Data loading       | Loads **all at once**           | Loads **one by one (on demand)** |
| Memory usage       | ❌ High                          | ✅ Low                            |
| Speed (initial)    | Slower (loads everything first) | Faster start                     |
| Access by index    | ✅ `docs[0]` works               | ❌ Not possible                   |
| Looping            | Works normally                  | Works once only                  |
| Reusability        | ✅ Can reuse multiple times      | ❌ Exhausted after one iteration  |
| Best for           | Small files                     | Large files / many files         |
| Debugging          | Easier                          | Slightly tricky                  |
| Real-world analogy | Download full movie 🎬          | Stream movie 📺                  |
 """

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader('file_name' , encoding='utf-8')

# .load()
docs1 = loader.load()
print(type(docs1)) #<class 'list'> because .load() returns a list of Documents
print(docs1[0].page_content)
print(docs1[0].metadata)


# .lazy_loading()
docs2 = loader.lazy_load() #<class 'generator'> because we used lazy_loading()
print(type(docs2))
for doc in docs2:
    print(type(doc)) ## Now after loading it will become a document object
    print(doc.page_content)
    print(doc.metadata)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# used when you have multiple files in a directory
loader = DirectoryLoader(
    path='directory_name',
    glob='*.pdf',
    loader_cls=PyPDFLoader
)

# glob parameter defines which files to load from the directory
# It follows Unix-style pattern matching

# Common glob patterns:
# "*.pdf"        → all PDF files in the directory
# "*.txt"        → all text files
# "*.docx"       → all Word files
# "*"            → all files (no filter)
# "*.*"          → all files with an extension

docs = loader.lazy_load()

for doc in docs:
    print(type(doc))
    print(doc.page_content)
    print(doc.metadata)
    

In [ ]:
""" | Method                             | How it splits                                   | Intelligence level | Pros                       | Cons                      | Best Use Case               |
| ---------------------------------- | ----------------------------------------------- | ------------------ | -------------------------- | ------------------------- | --------------------------- |
| **CharacterTextSplitter**          | Fixed characters (e.g. 200 chars)               | ❌ Low              | Simple, fast               | Breaks words/sentences    | Small demos, testing        |
| **RecursiveCharacterTextSplitter** | Paragraph → line → word → char fallback         | ✅ Medium           | Keeps structure better     | Slightly slower           | 🔥 Most common for RAG      |
| **TokenTextSplitter**              | Based on LLM tokens                             | ✅ Medium           | Matches model limits       | Needs tokenizer           | When using GPT/token limits |
| **Document-based (page/section)**  | Splits by document structure (pages, headings)  | ✅ Medium           | Preserves logical sections | Depends on loader quality | PDFs, DOCX, reports         |
| **Semantic Chunking**              | Splits based on meaning (embeddings similarity) | 🔥 High            | Context-aware chunks       | Expensive, complex        | High-quality RAG systems    |
| **SentenceSplitter**               | Splits by sentences                             | ✅ Medium           | Clean readable chunks      | May lose context          | QA systems, summarization   |
| **Markdown/Header Splitter**       | Splits by headings (`#`, `##`)                  | ✅ Medium           | Keeps hierarchy            | Only for structured docs  | Docs, blogs, notes          |
| **HTML Splitter**                  | Splits by tags                                  | ✅ Medium           | Good for web scraping      | HTML-dependent            | Websites, scraping          |
| **Code Splitter**                  | Splits by functions/classes                     | 🔥 High            | Preserves code logic       | Language-specific         | Code analysis tools         |
 """

## You can use all these by similar syntax of code given below

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
loader = TextLoader('cricket.txt' , encoding='utf-8')

docs = loader.load()
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=30 , separator='')
result = splitter.split_documents(docs)
print(len(result))

for chunk in result:
    print(chunk.page_content)
    print("\n" + "-"*50 + "\n")  # separator for clarity

5
Under the golden sun, the मैदान wakes,
A quiet pitch, where destiny takes shape.
Leather meets willow with a cracking sound,
Echoes of passion spread all around.

The bowler runs with fire in his stri

--------------------------------------------------

ler runs with fire in his stride,
Dreams of glory burning deep inside.
The batter stands, calm yet fierce,
Eyes on the ball, the silence he pierces.

A cover drive—smooth as a flowing stream,
Crowd er

--------------------------------------------------

as a flowing stream,
Crowd erupts, living the same dream.
A rising catch, a moment in the air,
Time stands still… will it be taken there?

Sweat and struggle in every play,
Fortunes change with each

--------------------------------------------------

ay,
Fortunes change with each passing day.
From dusty streets to stadium lights,
Cricket is more than just bat and fights.

It’s hope, it’s pride, it’s joy, it’s pain,
A game of heart played again and

------------------------------------

In [ ]:
""" | Feature                 | Normal Database            | Vector Store                  | Vector Database                          |
| ----------------------- | -------------------------- | ----------------------------- | ---------------------------------------- |
| Data type               | Structured (rows, columns) | Embeddings (vectors)          | Embeddings (vectors)                     |
| Storage format          | Tables (SQL/NoSQL)         | Key-value / simple index      | Optimized vector index (HNSW, IVF, etc.) |
| Query type              | Exact match (WHERE, JOIN)  | Similarity search             | Fast similarity + filtering              |
| Search method           | Keywords, filters          | Cosine similarity / distance  | Approximate nearest neighbor (ANN)       |
| Speed (semantic search) | ❌ Slow / not possible      | ⚠️ Okay for small data        | ✅ Very fast at scale                     |
| Scalability             | High (for structured data) | Limited                       | Very high (millions/billions vectors)    |
| Examples                | MySQL, MongoDB             | FAISS (local), Chroma (basic) | Pinecone, Weaviate, Milvus               |
| Use case                | Banking, apps, CRUD        | Small RAG apps                | Production AI systems                    |
| Indexing                | B-tree, hash               | Basic vector index            | Advanced ANN indexes                     |
| Semantic understanding  | ❌ No                       | ✅ Yes                         | ✅ Yes (optimized)                        |
 """

In [3]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

doc1 = Document(page_content='Woh bhi apne na huye' , metadata = {"team":"Lyrics1"})
doc2 = Document(page_content='Dil bhi gya hathon se' , metadata = {"team":"Lyrics2"})
doc3 = Document(page_content='Aise aane se toh behtar' , metadata = {"team":"Lyrics3"})
doc4 = Document(page_content='Tha na aana unka' , metadata = {"team":"Lyrics4"})
doc5 = Document(page_content='Unke andaaz e karam' , metadata = {"team":"Lyrics5"})

docs = [doc1 , doc2 , doc3 , doc4 , doc5]

vector_store = Chroma(embedding_function=embedding , persist_directory='my_chroma_db' , collection_name='sample')

# add documents
vector_store.add_documents(docs) # It add vectors to db and also assign unique IDs

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


['6539ead8-448e-4f00-9894-527867f0cc50',
 'abfeba02-4226-47b5-8439-48ab41ab3846',
 'e559383b-91b3-46a6-9f0a-01ac745e318c',
 '760090a7-8b84-4c66-8dae-76c57d7653f6',
 '7b841148-a8f8-49cf-85a0-fdec1bad1bb5']

In [4]:
# view documents
vector_store.get(include=['embeddings' , 'documents' , 'metadatas'])

{'ids': ['6539ead8-448e-4f00-9894-527867f0cc50',
  'abfeba02-4226-47b5-8439-48ab41ab3846',
  'e559383b-91b3-46a6-9f0a-01ac745e318c',
  '760090a7-8b84-4c66-8dae-76c57d7653f6',
  '7b841148-a8f8-49cf-85a0-fdec1bad1bb5'],
 'embeddings': array([[-0.06115789,  0.09732477,  0.03515448, ...,  0.06372225,
         -0.02879616, -0.01196219],
        [ 0.01962134,  0.09764971,  0.03375487, ...,  0.11202952,
          0.01199048, -0.09832095],
        [-0.05867844,  0.0819008 ,  0.02811179, ...,  0.08645378,
         -0.01326264, -0.05642597],
        [-0.08875249,  0.05899365, -0.06409108, ...,  0.06393117,
         -0.08246699, -0.05591628],
        [-0.0764655 ,  0.06856912,  0.01681596, ..., -0.02067927,
         -0.08988614, -0.06289605]]),
 'documents': ['Woh bhi apne na huye',
  'Dil bhi gya hathon se',
  'Aise aane se toh behtar',
  'Tha na aana unka',
  'Unke andaaz e karam'],
 'uris': None,
 'included': ['embeddings', 'documents', 'metadatas'],
 'data': None,
 'metadatas': [{'team': 'Lyr

In [ ]:
# search documents
vector_store.similarity_search(query='love song' , k=2) #k represents how much similar objects you want to retrieve
#Give me the top 2 most similar documents

[Document(id='abfeba02-4226-47b5-8439-48ab41ab3846', metadata={'team': 'Lyrics2'}, page_content='Dil bhi gya hathon se'),
 Document(id='6539ead8-448e-4f00-9894-527867f0cc50', metadata={'team': 'Lyrics1'}, page_content='Woh bhi apne na huye')]

In [6]:
#search with score
vector_store.similarity_search_with_score(query="love", k=2)

[(Document(id='abfeba02-4226-47b5-8439-48ab41ab3846', metadata={'team': 'Lyrics2'}, page_content='Dil bhi gya hathon se'),
  1.4646462202072144),
 (Document(id='6539ead8-448e-4f00-9894-527867f0cc50', metadata={'team': 'Lyrics1'}, page_content='Woh bhi apne na huye'),
  1.5584560632705688)]

In [7]:
#meta-data filtering
vector_store.similarity_search_with_score(query='' , filter={'team':"Lyrics5"})

[(Document(id='7b841148-a8f8-49cf-85a0-fdec1bad1bb5', metadata={'team': 'Lyrics5'}, page_content='Unke andaaz e karam'),
  1.6019296646118164)]

In [8]:
# update documents
updated_doc1 = Document(page_content='Ho ho mai tere te mar gyi sajna re tu smjha na' , metadata = {'team':'Lyrics6'})
vector_store.update_document(document_id='7b841148-a8f8-49cf-85a0-fdec1bad1bb5' , document=updated_doc1)

In [ ]:
#delete documents
vector_store.delete(ids=['7b841148-a8f8-49cf-85a0-fdec1bad1bb5'])

In [ ]:
## Retrievres

## 1. Data sources based Retrivers

""" | Retriever                  | What it does                                   | Use case                  |
| -------------------------- | ---------------------------------------------- | ------------------------- |
| **Vector Store Retriever** | Searches your embeddings (Chroma, FAISS, etc.) | 🔥 Most common (RAG apps) |
| **Wikipedia Retriever**    | Fetches info from Wikipedia                    | Quick factual Q&A         |
| **Arxiv Retriever**        | Fetches research papers                        | Academic / research apps  |
| **Web Retriever**          | Searches web (via APIs)                        | Real-time info            |
| **Multi-Vector Retriever** | Uses multiple embeddings per doc               | Advanced RAG              |
| **Self-Query Retriever**   | Converts query → metadata filters              | Smart filtering           |
 """


### 2. Search strategy Based retriever

""" | Retriever                            | What it does               | Problem it solves         |
| ------------------------------------ | -------------------------- | ------------------------- |
| **MMR (Max Marginal Relevance)**     | Returns diverse results    | Avoids duplicate chunks   |
| **Contextual Compression Retriever** | Removes irrelevant text    | Reduces noise/token usage |
| **Multi-Query Retriever**            | Generates multiple queries | Improves recall           |
| **Time-weighted Retriever**          | Prefers recent data        | Time-sensitive apps       |
| **Parent-Document Retriever**        | Retrieves bigger context   | Avoids broken chunks      |
 """

In [3]:
from langchain_community.retrievers import WikipediaRetriever
retriever = WikipediaRetriever(top_k_results=2 , lang='en')
query='World war'
docs = retriever.invoke(query)
for i,docs in enumerate(docs):
    print(f"Result {i+1}")
    print(f"Content \n {docs.page_content}")

Result 1
Content 
 A world war is an international conflict that involves most or all of the world's major powers. Conventionally, the term is reserved for the two major international conflicts that occurred during the first half of the 20th century: World War I (1914–1918) and World War II (1939–1945). Some historians have also characterized other global conflicts as world wars, such as the Cold War and the war on terror.


== Etymology ==
The Oxford English Dictionary had cited the first known usage in the English language to a Scottish newspaper, The People's Journal, in 1848: "A war among the great powers is now necessarily a world-war." The term "world war" is used by Karl Marx and his associate, Friedrich Engels, in a series of articles published around 1850 called The Class Struggles in France. Rasmus B. Anderson in 1889 described an episode in Teutonic mythology as a "world war" (Swedish: världskrig), justifying this description by a line in an Old Norse epic poem, "Völuspá: fo

In [1]:
## Vector store retriever

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

doc1 = Document(page_content='Netanyahu will be in hell' , metadata = {"team":"Lyrics1"})
doc2 = Document(page_content='Love is which we never get' , metadata = {"team":"Thought"})
doc3 = Document(page_content='Virat kohli is a batsman' , metadata = {"team":"Cricket"})
doc4 = Document(page_content='I love being baller' , metadata = {"team":"Cricket"})

doc = [doc1 , doc2 , doc3 , doc4]

vector_store = Chroma.from_documents(documents=doc , embedding=embedding , persist_directory='my_chroma_db' , collection_name='my_collection')
retriever = vector_store.as_retriever(search_kwargs={"k":2})
query = "Related to cricket"
result = retriever.invoke(query)
for i in result:
    print(i)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


page_content='Virat kohli is a batsman' metadata={'team': 'Cricket'}
page_content='I love being baller' metadata={'team': 'Cricket'}


In [ ]:
## Maximal Marginal relevance ( " How can we pick results that are not only relevant to the query but also differenet from each other")

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

docs = [
    Document(page_content="Virat Kohli is a batsman"),
    Document(page_content="Virat Kohli is a great cricketer"),
    Document(page_content="Virat Kohli plays cricket"),
    Document(page_content="Football is a popular sport"),
    Document(page_content="Basketball is played worldwide"),
]

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="my_chroma_db",
    collection_name="mmr_demo"
)

query = "cricket player"

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2, #give me top 2 results
        "fetch_k": 5 , #First find top 5 similar docs, then choose best 2 using MMR
        "lambda_mult": 0 #Controls balance between relevance vs diversity ( more close to 0 means more diversity)
    }
)

mmr_results = mmr_retriever.invoke(query)

for i in mmr_results:
    print(i)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


page_content='Virat Kohli plays cricket'
page_content='Football is a popular sport'


In [1]:
## MultiQueryretriver ( It uses a llm o generate a multiple sub query for the query provided and then gives us relevant result)

from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

docs = [
    # Cricket (same idea, different wording)
    Document(page_content="Virat Kohli is one of the best batsmen in cricket"),
    Document(page_content="Kohli is a famous Indian cricket player"),
    Document(page_content="A batsman named Virat plays for India"),

    # Football
    Document(page_content="Lionel Messi is a legendary football player"),
    Document(page_content="Messi plays soccer and is from Argentina"),

    # AI / Tech
    Document(page_content="Artificial Intelligence is transforming industries"),
    Document(page_content="AI is changing the future of technology"),
    Document(page_content="Machine learning is a part of artificial intelligence"),

    # Random noise
    Document(page_content="I love eating pizza on weekends"),
    Document(page_content="The weather is very pleasant today"),
]

vector_store = FAISS.from_documents(documents=docs , embedding=embedding)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 4,
            "fetch_k": 10,
            "lambda_mult":0.5
        }
    ),
    llm=ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)
)

query = "Virat Kohli cricket player"
multiquery_result = multiquery_retriever.invoke(query)

for j in multiquery_result:
    print(j)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


page_content='Machine learning is a part of artificial intelligence'
page_content='A batsman named Virat plays for India'
page_content='The weather is very pleasant today'
page_content='Lionel Messi is a legendary football player'
page_content='Virat Kohli is one of the best batsmen in cricket'
page_content='AI is changing the future of technology'
page_content='Kohli is a famous Indian cricket player'
page_content='I love eating pizza on weekends'


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

docs = [
    Document(page_content="Virat Kohli is one of the best batsmen in cricket"),
    Document(page_content="Kohli is a famous Indian cricket player"),
    Document(page_content="A batsman named Virat plays for India"),
    Document(page_content="Lionel Messi is a legendary football player"),
    Document(page_content="Artificial Intelligence is transforming industries"),
    Document(page_content="I love eating pizza on weekends"),
]

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding
)

base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 6}  # fetch top 6 similar docs (may include noise)
)


llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

compressor = LLMChainExtractor.from_llm(llm)
# This uses LLM to EXTRACT only relevant parts from documents

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

query = "Virat Kohli cricket player"

results = compression_retriever.invoke(query)

for doc in results:
    print(doc)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


page_content='Virat Kohli is one of the best batsmen in cricket'
page_content='Kohli is a famous Indian cricket player'
page_content='A batsman named Virat plays for India'
page_content='Lionel Messi is a legendary football player'
